## DIAZOTROPS DATA
This notebook aims to explore the initial data sources for the environmental data

In [441]:
#we import all they key libraries needed in this notebook
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# First dataset
This dataset is from the paper [Global oceanic diazotroph database version 2 and
elevated estimate of global oceanic N2 fixation](https://essd.copernicus.org/articles/15/3673/2023/essd-15-3673-2023-assets.html). It contains several datasets as sources. In particular there is a file dates 2023 and one dates 2024. It is written by the authors that the latter version has some erros fixed so I opted to us it here.

The first sheet I decided to examine as the xls file contains several is the cell count sheet. It clearly has coordinates that we can use to join it with environmental data, but the floating point values would not match perfectly with the NCEI dataset(there every number ends with 0.5). This can cause issues when joining as inner join can yield an empty result and the normal join based on indecies can create a lot of empty cells with each row containing only environmental or cell data but very rarely both. 

Thus, I am considering rounding the coordinates in both datasets in order to have higher chances of there being matches between coordinates.

In [442]:
dzdb_2024_cc = pd.read_csv("./csv/DiazotrophsDatabase-20240109.csv")
print(dzdb_2024_cc.shape)
dzdb_2024_cc.head()

(6765, 24)


,SOURCE: Related article,METHODS: Sampling/Analysis,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,DEPTH (m),Trichodesmium Colonies (x103 m-3),Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),UCYN-A Cells (x106 m-3),...,Richelia intracellularis (x106 m-3),Calothrix Associated Species,Calothrix intracellularis (x106 m-3),Temperature (˚C),Salinity (PSU),Nitrate (µM),Phosphate (µM),Fe (nM),Chlorophyll (mg m-3),Notes
0,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,7/9/1974,13.2,-59.7,0,NaN,NaN,0.67,NaN,...,NaN,NaN,NaN,27.5,32.67,NaN,NaN,NaN,0.12,NaN
1,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,5,NaN,NaN,0.58,NaN,...,NaN,NaN,NaN,27.5,32.79,NaN,NaN,NaN,NaN,NaN
2,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,10,NaN,NaN,0.69,NaN,...,NaN,NaN,NaN,27.5,32.54,NaN,NaN,NaN,0.20,NaN
3,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,15,NaN,NaN,0.51,NaN,...,NaN,NaN,NaN,27.5,32.80,NaN,NaN,NaN,0.14,NaN
4,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,25,NaN,NaN,0.43,NaN,...,NaN,NaN,NaN,27.0,33.48,NaN,NaN,NaN,0.19,NaN


## Null values of the dataset
When opening the dataset I noticed the abundance of empty cells I decided that one of the first check we need to perform is for empty values in columns that interest us the most. Clearly the best chances of producing the valid prediction we will have with "Trichodesmium Total Trichomes (x106 m-3) ", and the other 2 Trichodesmium column. 

However, the very low count of non empty cells for UCYN bacteria makes me doubtful if this will be enough.

In [443]:
print(dzdb_2024_cc.notna().sum())
dzdb_2024_cc.describe()

SOURCE: Related article                       84
METHODS: Sampling/Analysis                  6765
DATE (yyyy-mm-dd)                           6765
LATITUDE                                    6765
LONGITUDE                                   6765
DEPTH (m)                                   6765
Trichodesmium Colonies (x103 m-3)           1225
Trichodesmium Free trichomes (x106 m-3)     1541
Trichodesmium Total Trichomes (x106 m-3)    5681
UCYN-A Cells (x106 m-3)                       84
UCYN-B Cells (x106 m-3)                        4
UCYN-C Cells (x106 m-3)                        0
UCYN Cells (x106 m-3)                         81
Richelia Associated Species                 1704
Richelia intracellularis (x106 m-3)         2748
Calothrix Associated Species                  52
Calothrix intracellularis  (x106 m-3)         87
Temperature (˚C)                            1360
Salinity (PSU)                              1589
Nitrate (µM)                                 774
Phosphate (µM)      

,LATITUDE,LONGITUDE,Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),UCYN-B Cells (x106 m-3),UCYN-C Cells (x106 m-3),UCYN Cells (x106 m-3),Richelia intracellularis (x106 m-3),Calothrix intracellularis (x106 m-3),Temperature (˚C),Salinity (PSU),Fe (nM)
count,6765.00000,6765.000000,1541.000000,5681.000000,4.000000,0.0,81.000000,2748.000000,87.000000,1360.000000,1589.000000,19.000000
mean,11.89643,13.995217,9.586801,22.389078,92.556000,NaN,29.146432,3.170271,0.048326,23.582963,35.266281,0.269861
std,20.70263,103.349535,156.331984,688.149460,118.101372,NaN,94.227838,23.973336,0.268968,4.877795,1.659966,0.425774
min,-64.34000,-180.000000,0.000000,0.000000,0.224000,NaN,0.400000,0.000000,0.000000,10.310000,-9.000000,0.000009
25%,0.00000,-59.700000,0.000000,0.000000,0.760250,NaN,2.939000,0.000000,0.000000,20.512500,34.650000,0.000214
50%,13.25000,-19.980000,0.000000,0.000000,61.099500,NaN,5.540000,0.000000,0.000000,24.405000,35.300000,0.000750
75%,29.36000,123.070000,0.045000,0.070000,152.895250,NaN,26.100000,0.014825,0.000750,26.982500,36.170000,0.420000
max,78.25000,179.610000,4700.000000,44200.000000,247.801000,NaN,819.000000,766.000000,1.903400,126.190000,38.580000,1.560000


The second sheet we can examine is the integral cell count sheet which I saved as a separate csv file. We perform the same exact operations with it to see the overall statistics.

In [444]:
dzdb_2024_cci = pd.read_csv("./csv/DiazotrophsDatabase-20240109_cellcount_int.csv")
print(dzdb_2024_cci.shape)
dzdb_2024_cci.head()

(1411, 21)


,SOURCE: Related article,SOURCE: Related article.1,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),UCYN Cells (x106 m-2),...,Richelia intracellularis (x106 m-2),Calothrix Associated Species,Calothrix intracellularis (x106 m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2),Notes
0,"Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-26,41.5,-9.32,65.0,NaN,0.000,0.000,NaN,...,NaN,NaN,NaN,18.60,35.70,0.61,0.06,NaN,0.15,2-3 tows performed with 50 µm net
1,NaN,Standard Light Microscopy,2009-07-27,41.5,-10.58,60.0,NaN,0.002,0.002,NaN,...,NaN,NaN,NaN,19.75,35.87,0.13,0.02,NaN,NaN,2-3 tows performed with 50 µm net
2,NaN,Standard Light Microscopy,2009-07-28,41.5,-12.26,62.0,NaN,0.003,0.003,NaN,...,NaN,NaN,NaN,19.81,35.99,0.05,0.02,NaN,0.02,2-3 tows performed with 50 µm net
3,NaN,Standard Light Microscopy,2009-07-30,41.5,-15.29,67.0,NaN,0.001,0.001,NaN,...,NaN,NaN,NaN,19.75,35.96,0.09,0.02,NaN,NaN,2-3 tows performed with 50 µm net
4,NaN,Standard Light Microscopy,2009-07-31,41.5,-17.18,53.0,NaN,0.007,0.007,NaN,...,NaN,NaN,NaN,18.95,35.87,0.05,0.03,NaN,0.40,2-3 tows performed with 50 µm net


In [445]:
print(dzdb_2024_cci.notna().sum())
dzdb_2024_cci.describe()

SOURCE: Related article                       47
SOURCE: Related article.1                   1411
DATE (yyyy-mm-dd)                           1411
LATITUDE                                    1411
LONGITUDE                                   1411
INTEGRAL DEPTH (m)                          1411
Trichodesmium Colonies (x103 m-2)            393
Trichodesmium Free trichomes (x106 m-2)      522
Trichodesmium Total Trichomes (x106 m-2)    1297
UCYN Cells (x106 m-2)                         19
Richelia Associated Species                  406
Richelia intracellularis (x106 m-2)          435
Calothrix Associated Species                   8
Calothrix intracellularis  (x106 m-2)         17
Surface Temperature (˚C)                     459
Surface Salinity (PSU)                       469
Surface Nitrate (µM)                         228
Surface Phosphate (µM)                       276
Surface Fe (nM)                              120
Chlorophyll  (mg m-2)                        173
Notes               

,LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),UCYN Cells (x106 m-2),Richelia intracellularis (x106 m-2),Calothrix intracellularis (x106 m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2)
count,1411.000000,1411.000000,1411.000000,393.000000,522.000000,1297.000000,19.000000,435.000000,17.000000,459.000000,469.000000,228.000000,276.000000,120.000000,173.000000
mean,16.946736,9.103940,109.081148,28.113821,10.608421,10.391441,1687.775158,538.423777,19.245118,26.643268,34.611748,0.128114,0.394493,1.612167,13.301908
std,16.696257,106.124185,71.222798,86.381629,45.076470,34.917039,2035.345391,2646.232524,41.391709,5.222897,2.121061,0.330944,0.780058,1.747653,8.197072
min,-40.500000,-180.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,15.080000,22.770000,0.000000,0.000000,0.090000,0.020000
25%,11.020000,-59.700000,57.000000,0.000000,0.058350,0.035400,595.629000,0.112500,0.000000,25.960000,33.960000,0.020000,0.020000,0.580000,5.880000
50%,21.571203,-30.740000,98.000000,0.000000,0.572150,0.600000,1015.000000,0.998000,0.000000,27.050000,35.200000,0.050000,0.050000,1.220000,12.700000
75%,28.500000,122.600000,175.000000,7.629000,2.954350,5.700000,1781.500000,4.979889,8.678000,27.900000,36.000000,0.065000,0.245000,2.170000,19.610000
max,41.840000,201.170000,1250.000000,586.683000,660.000000,660.000000,8208.600000,26039.992000,144.445000,126.190000,37.600000,3.010000,4.240000,11.760000,35.110000


# Second dataset
This is the dataset from the  [Database of diazotrophs in global ocean: abundance, biomass and nitrogen fixation rates](https://doi.pangaea.de/10.1594/PANGAEA.774851)

We can see that the not null value count is again the best for Trichodesmium with other bacteria having significantly less. 

Moreover, the same holds for coordinates that should be rounded before joining the datasets together into a single sheet.

In [446]:
#here we open the dataset of the other database for cell count
maredat_cc = pd.read_csv("./csv/MAREDAT_diazotroph_cellcount.csv")
print(maredat_cc.shape)
maredat_cc.head()

(3842, 30)


,SOURCE: Data,SOURCE: Related article,METHODS: Sampling/Analysis,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,DEPTH (m),Trichodesmium Colonies (x103 m-3),Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),...,Calothrix Biomass Conversion factor (mg C/106 cells),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3),Total Diazotrophic Biomass (mg C m-3),Temperature (˚C),Salinity (PSU),Nitrate (µM),Phosphate (µM),Fe (nM),Chlorophyll (mg m-3),Notes
0,NaN,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,1974-07-09,13.2,-59.7,0,NaN,NaN,0.668,...,NaN,NaN,20.04,27.5,32.67,NaN,NaN,NaN,0.12,NaN
1,NaN,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,1974-07-09,13.2,-59.7,5,NaN,NaN,0.580,...,NaN,NaN,17.40,27.5,32.79,NaN,NaN,NaN,NaN,NaN
2,NaN,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,1974-07-09,13.2,-59.7,10,NaN,NaN,0.685,...,NaN,NaN,20.55,27.5,32.54,NaN,NaN,NaN,0.20,NaN
3,NaN,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,1974-07-09,13.2,-59.7,15,NaN,NaN,0.511,...,NaN,NaN,15.33,27.5,32.80,NaN,NaN,NaN,0.14,NaN
4,NaN,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,1974-07-09,13.2,-59.7,25,NaN,NaN,0.432,...,NaN,NaN,12.96,27.0,33.48,NaN,NaN,NaN,0.19,NaN


In [447]:
#we want to see null value count as well as the properties of the data again
print(maredat_cc.notna().sum())
maredat_cc.describe()

SOURCE: Data                                                    2356
SOURCE: Related article                                         3539
METHODS:                  Sampling/Analysis                     3842
DATE (yyyy-mm-dd)                                               3842
LATITUDE                                                        3842
LONGITUDE                                                       3842
DEPTH (m)                                                       3842
Trichodesmium Colonies (x103 m-3)                                954
Trichodesmium Free trichomes (x106 m-3)                          613
Trichodesmium Total Trichomes (x106 m-3)                        3088
Trichodesmium Biomass Conversion factor (mg C/106 trichomes)    3088
Trichodesmium Biomass (mg C m-3)                                3139
UCYN Cells (x106 m-3)                                              0
UCYN Biomass Conversion factor (mg C/106 cells)                    0
UCYN Biomass (mg C m-3)           

,LATITUDE,LONGITUDE,DEPTH (m),Trichodesmium Colonies (x103 m-3),Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),Trichodesmium Biomass (mg C m-3),UCYN Cells (x106 m-3),UCYN Biomass Conversion factor (mg C/106 cells),UCYN Biomass (mg C m-3),...,Calothrix Cells (x106 m-3),Calothrix Biomass Conversion factor (mg C/106 cells),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3),Total Diazotrophic Biomass (mg C m-3),Temperature (˚C),Salinity (PSU),Nitrate (µM),Phosphate (µM),Fe (nM),Chlorophyll (mg m-3)
count,3842.000000,3842.000000,3842.000000,954.000000,613.000000,3088.000000,3.139000e+03,0.0,0.0,0.0,...,48.000000,4.800000e+01,1564.000000,3.842000e+03,932.000000,1162.000000,735.000000,702.000000,16.000000,931.000000
mean,11.445333,-25.505966,39.852941,233.206339,0.051108,14.610203,4.321215e+02,NaN,NaN,NaN,...,0.127771,1.000000e-02,0.113933,3.530993e+02,23.781921,35.086015,0.321936,0.099675,0.170460,0.233856
std,19.110383,86.046067,51.609583,7155.096371,0.123636,795.393852,2.366718e+04,NaN,NaN,NaN,...,0.300774,5.259243e-18,0.591384,2.139262e+04,5.273862,1.822386,0.956232,0.118878,0.303938,0.279054
min,-51.930000,-180.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,...,0.000000,1.000000e-02,0.000000,0.000000e+00,10.310000,-9.000000,-0.002000,0.000000,0.000009,0.000000
25%,4.257500,-67.010000,5.000000,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,...,0.000000,1.000000e-02,0.000000,0.000000e+00,20.300000,34.512500,0.030000,0.020000,0.000205,0.080000
50%,13.200000,-54.000000,15.000000,0.021000,0.000500,0.004000,1.410000e-01,NaN,NaN,NaN,...,0.000000,1.000000e-02,0.000000,3.000000e-02,25.200000,35.150000,0.080000,0.060000,0.000339,0.150000
75%,27.470000,39.280000,60.000000,0.724750,0.042200,0.135250,4.620000e+00,NaN,NaN,NaN,...,0.042500,1.000000e-02,0.000600,2.589250e+00,27.032500,35.990000,0.180000,0.116500,0.210750,0.250000
max,49.320000,179.480000,350.000000,221000.000000,1.613100,44200.000000,1.326000e+06,NaN,NaN,NaN,...,1.527500,1.000000e-02,8.132800,1.326000e+06,126.190000,37.630000,9.350000,0.925000,0.971000,1.820000


Here we show the cell count integral data of the dataset

In [448]:
maredat_cci = pd.read_csv("./csv/MAREDAT_diazotroph_cellcount_int.csv")
print(maredat_cci.shape)
maredat_cci.head()

(692, 30)


,SOURCE: Data,SOURCE: Related article,METHODS: Sampling/Analysis,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),...,Calothrix Biomass Conversion factor (mg C/106 cells),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-2),Total Diazotrophic Biomass (mg C m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2),Notes
0,"Benavides, M., Instituto de Oceanografía y Cam...","Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-26,41.5,-9.32,65,NaN,0.0001,0.0001,...,NaN,NaN,0.003,18.60,35.70,0.61,0.06,NaN,0.15,2-3 tows performed with 50 µm net
1,"Benavides, M., Instituto de Oceanografía y Cam...","Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-27,41.5,-10.58,60,NaN,0.0016,0.0016,...,NaN,NaN,0.047,19.75,35.87,0.13,0.02,NaN,NaN,2-3 tows performed with 50 µm net
2,"Benavides, M., Instituto de Oceanografía y Cam...","Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-28,41.5,-12.26,62,NaN,0.0027,0.0027,...,NaN,NaN,0.082,19.81,35.99,0.05,0.02,NaN,0.02,2-3 tows performed with 50 µm net
3,"Benavides, M., Instituto de Oceanografía y Cam...","Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-30,41.5,-15.29,67,NaN,0.0007,0.0007,...,NaN,NaN,0.021,19.75,35.96,0.09,0.02,NaN,NaN,2-3 tows performed with 50 µm net
4,"Benavides, M., Instituto de Oceanografía y Cam...","Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-31,41.5,-17.18,53,NaN,0.0067,0.0067,...,NaN,NaN,0.202,18.95,35.87,0.05,0.03,NaN,0.40,2-3 tows performed with 50 µm net


In [449]:
print(maredat_cci.notna().sum())
maredat_cci.describe()

SOURCE: Data                                                    440
SOURCE: Related article                                         652
METHODS:                  Sampling/Analysis                     692
DATE (yyyy-mm-dd)                                               692
LATITUDE                                                        692
LONGITUDE                                                       692
INTEGRAL DEPTH (m)                                              692
Trichodesmium Colonies (x103 m-2)                               141
Trichodesmium Free trichomes (x106 m-2)                          85
Trichodesmium Total Trichomes (x106 m-2)                        620
Trichodesmium Biomass Conversion factor (mg C/106 trichomes)    620
Trichodesmium Biomass (mg C m-2)                                626
UCYN Cells (x106 m-2)                                             0
UCYN Biomass Conversion factor (mg C/106 cells)                   0
UCYN Biomass (mg C m-2)                         

,LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),Trichodesmium Biomass (mg C m-2),UCYN Cells (x106 m-2),UCYN Biomass Conversion factor (mg C/106 cells),UCYN Biomass (mg C m-2),...,Calothrix Cells (x106 m-2),Calothrix Biomass Conversion factor (mg C/106 cells),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-2),Total Diazotrophic Biomass (mg C m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2)
count,692.00000,692.000000,692.000000,141.000000,85.000000,620.000000,626.000000,0.0,0.0,0.0,...,17.000000,17.00,288.000000,692.000000,378.000000,388.000000,162.000000,219.000000,62.000000,173.000000
mean,17.73172,-17.184697,102.147399,76.743794,0.762852,11.887856,306.715580,NaN,NaN,NaN,...,96.225871,0.01,32.975242,291.186176,26.502090,34.749485,0.153111,0.077087,1.666344,13.301908
std,11.61591,99.861774,59.209493,130.841014,1.198999,28.528307,684.586352,NaN,NaN,NaN,...,206.958794,0.00,149.699116,662.270521,5.719384,1.969303,0.388230,0.147071,1.719042,8.197072
min,-33.34000,-180.000000,4.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,...,0.000000,0.01,0.000000,0.000000,15.080000,22.770000,0.000000,0.000000,0.180000,0.020000
25%,13.03500,-67.525000,60.000000,3.169000,0.006700,0.118600,3.474000,NaN,NaN,NaN,...,0.000000,0.01,0.005000,1.200000,25.765000,34.140000,0.015000,0.016000,0.610000,5.880000
50%,18.27500,-51.405000,100.000000,19.533000,0.141000,1.550000,43.305000,NaN,NaN,NaN,...,0.000000,0.01,0.044350,31.231000,26.915000,35.220000,0.035000,0.031000,1.303575,12.700000
75%,25.49250,115.500000,150.000000,60.927000,1.101200,10.192950,283.608750,NaN,NaN,NaN,...,43.392200,0.01,0.295975,251.385000,27.810000,35.990000,0.094750,0.068000,2.201198,19.610000
max,41.84000,201.170000,350.000000,586.683000,7.559600,320.347600,8953.200000,NaN,NaN,NaN,...,722.225600,0.01,1302.000000,8953.200000,126.190000,37.600000,3.010000,1.360000,11.759642,35.110000


# Combining the dataframes together into 1
This parts aims to combine the data into 1 based on each feature.

In [450]:
#there are 2 sets of columns, 1 for key values and 1 for data columns both of which we want to keep
key_col = ['LATITUDE', 'LONGITUDE', 'DEPTH (m)']
data_cols = ["Trichodesmium Colonies (x103 m-3)","Trichodesmium Free trichomes (x106 m-3)","Trichodesmium Total Trichomes (x106 m-3)"]

Initial data before the transformations of the tables

In [451]:
#we want to take only values for each feature that are not null, but first I wanted to check the column names in case they differ
print(dzdb_2024_cc.info())
print(maredat_cc.info())

print("Intital shapes; set 1: {0}; set 2: {1}".format(dzdb_2024_cc.shape, maredat_cc.shape))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6765 entries, 0 to 6764
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   SOURCE: Related article                   84 non-null     object 
 1   METHODS: Sampling/Analysis                6765 non-null   object 
 2   DATE (yyyy-mm-dd)                         6765 non-null   object 
 3   LATITUDE                                  6765 non-null   float64
 4   LONGITUDE                                 6765 non-null   float64
 5   DEPTH (m)                                 6765 non-null   object 
 6   Trichodesmium Colonies (x103 m-3)         1225 non-null   object 
 7   Trichodesmium Free trichomes (x106 m-3)   1541 non-null   float64
 8   Trichodesmium Total Trichomes (x106 m-3)  5681 non-null   float64
 9   UCYN-A Cells (x106 m-3)                   84 non-null     object 
 10  UCYN-B Cells (x106 m-3)             

### Object columns
A lot of columns are labeled as object despite containing numeric values

### We don't need every column
Not every column is actually needed by us. Thus we have to decide which columns to keep and only store those.

In [452]:
#this function removes all not specified columns from the dataframe
def removeRed(keepers, df):
    cols = set(df.columns) - set(keepers)
    df.drop(columns=list(cols), inplace=True)

#redundant columns are removed before joining
removeRed(key_col+data_cols, dzdb_2024_cc)
removeRed(key_col+data_cols, maredat_cc)

In [453]:
#this function converts object columns to numeric
def objToNum(df):
    object_columns = df.select_dtypes(include='object').columns.tolist()
    for col in object_columns:
        df[col]=pd.to_numeric(df[col], errors="coerce")

#we want to convert all object columns to numeric values
objToNum(dzdb_2024_cc)
objToNum(maredat_cc)


### Desired depth is only below 25m
The depth range might be more than we want, so we would have to filter it as well.

We can clearly see a significant reduction in values count when we filter based on depth. However, we should also check if coordinates repeat. Because we might have several records from the same site, where the data should be averaged.

In [454]:
#now we filter the data based on the depth
dzdb_2024_cc_depth_mask = (dzdb_2024_cc["DEPTH (m)"]<=25)
maredat_cc_depth_mask = (maredat_cc["DEPTH (m)"]<=25)

dzdb_2024_cc_df = dzdb_2024_cc[dzdb_2024_cc_depth_mask]
maredat_cc_df = maredat_cc[maredat_cc_depth_mask]

print("Depth filtered shapes; set 1: {0}; set 2: {1}".format(dzdb_2024_cc_df.shape, maredat_cc_df.shape))

Depth filtered shapes; set 1: (3840, 6); set 2: (2333, 6)


In [455]:
#there appears to be a lot of duplicate locations in the dataset
print(dzdb_2024_cc_df.duplicated(subset=['LATITUDE', 'LONGITUDE']).sum())
print(maredat_cc_df.duplicated(subset=['LATITUDE', 'LONGITUDE']).sum())

2003
1114


In [456]:
#Now I want to average the dupliactes before joining
dzdb_2024_cc_df.groupby(by=['LATITUDE', 'LONGITUDE']).mean().notnull().sum()

DEPTH (m)                                   1837
Trichodesmium Colonies (x103 m-3)            550
Trichodesmium Free trichomes (x106 m-3)      861
Trichodesmium Total Trichomes (x106 m-3)    1631
dtype: int64